# Homework 13: Productization

This self-contained submission trains a two-feature regression model, saves it with joblib, and proves that a Flask API can serve both JSON and path-parameter calls.

## 1. Train, save, and reload the model

The model directory is created before saving. The reload check proves the serialized artifact is usable.

In [1]:
from pathlib import Path
import os

import joblib
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

X, y = make_regression(n_samples=100, n_features=2, noise=0.1, random_state=42)
model = LinearRegression().fit(X, y)
os.makedirs("model", exist_ok=True)
joblib.dump(model, "model/model.pkl")
reloaded = joblib.load("model/model.pkl")
print("saved to model/model.pkl")
print("prediction from reloaded model:", reloaded.predict([[0.1, 0.2]])[0])

saved to model/model.pkl
prediction from reloaded model: 23.589611712973284


## 2. API implementation

`app.py` loads the model once at startup. Both routes return JSON, and malformed values produce JSON HTTP 400 responses rather than tracebacks.

In [2]:
from pathlib import Path

app_path = Path("app.py")
assert app_path.is_file()
print(app_path.read_text(encoding="utf-8"))

"""Stage13 homework Flask prediction API."""

from pathlib import Path

import joblib
import numpy as np
from flask import Flask, jsonify, request

MODEL_PATH = Path(__file__).resolve().parent / "model" / "model.pkl"
model = joblib.load(MODEL_PATH)
app = Flask(__name__)


def parse_features(values):
    """Validate exactly two finite numeric inputs for the saved regression model."""
    if not isinstance(values, list) or len(values) != 2:
        raise ValueError("features must be a list of exactly 2 numbers")
    try:
        array = np.asarray(values, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError("features must contain only numbers") from exc
    if not np.isfinite(array).all():
        raise ValueError("features must contain only finite numbers")
    return array.tolist()


def error_response(message):
    return jsonify({"error": message}), 400


@app.post("/predict")
def predict_post():
    body = request.get_json(silent=True) or {}
    if "featu

## 3. Requests evidence

The cell launches a local process, calls both routes with `requests`, makes one deliberately bad call, then stops the process. Its visible output is the API test evidence.

In [3]:
import subprocess
import sys
import time

import requests

server = subprocess.Popen([sys.executable, "app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
base = "http://127.0.0.1:5001"
try:
    for _ in range(20):
        try:
            requests.get(base + "/predict/0.1/0.2", timeout=0.5)
            break
        except requests.ConnectionError:
            time.sleep(0.2)
    post = requests.post(base + "/predict", json={"features": [0.1, 0.2]}, timeout=5)
    get = requests.get(base + "/predict/0.1/0.2", timeout=5)
    bad = requests.get(base + "/predict/abc/0.2", timeout=5)
    print("POST /predict:", post.status_code, post.json())
    print("GET /predict/0.1/0.2:", get.status_code, get.json())
    print("GET /predict/abc/0.2:", bad.status_code, bad.json())
    assert post.status_code == get.status_code == 200
    assert bad.status_code == 400 and "error" in bad.json()
finally:
    server.terminate()
    server.wait(timeout=5)

POST /predict: 200 {'prediction': 23.589611712973284}
GET /predict/0.1/0.2: 200 {'prediction': 23.589611712973284}
GET /predict/abc/0.2: 400 {'error': 'features must contain only numbers'}


## 4. Handoff

See `README.md` for the exact server command, copy-pasteable examples for both routes, and the bad-input contract.